# Customer Purchase Behaviour and Retention Analysis
### Who buys, who returns, and what drives loyalty at Immaculate Vibes

**The business problem.** Immaculate Vibes wins customers through several channels (TikTok, WhatsApp, Instagram, Referral and Walk-in), but it doesn't yet know which customers come back, which channels bring in loyal buyers, or how much each type of customer is really worth. This notebook answers those questions so that retention spending can go where it pays off.

**How the notebook is organised**

| Part | What it does |
|---|---|
| 1. Setup | Loads the tools and sets the look of the charts |
| 2. Data preparation | Checks quality, separates cancelled/returned orders, builds helper columns |
| 3. Module 1 – Customer base | How many customers do we have, and how many buy again? |
| 4. Module 2 – RFM segmentation | Scores every customer on Recency, Frequency and Monetary value |
| 5. Module 3 – Channels | Which channels bring in loyal, valuable customers? |
| 6. Module 4 – Demographics & geography | Who is the most valuable customer? |
| 7. Module 5 – Timing | When do people buy? |
| 8. Module 6 – CLV & retention | What is a repeat customer worth, and who is slipping away? |
| 9. Wrap-up | Executive summary, recommendations, limitations |
| **10. Deployment** | **Turns the analysis into a weekly Retention Action Engine: parity check, back-test against a baseline, ROI scenarios, action list, monitoring plan and responsible-use rules** |

*Currency: prices look like Kenyan shillings (KES), given the M-PESA payments, so I label money as KES.*

> **Deployment version.** Sections 1–9 are the original exploratory analysis, unchanged except for configurable paths. Section 10 is new: it imports the production code in `iv_retention/pipeline.py` and proves it gives the same answers as this notebook.

## 1. Setup

First we load the libraries we need: **pandas** for working with the data, **matplotlib/seaborn** for charts. We also set a few display defaults so numbers are readable (thousands separators, two decimals) and every chart is saved into a `figures/` folder, so you can drop them straight into a report or slide deck.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")

# --- Display settings -------------------------------------------------------
pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

# --- Colours used consistently throughout ----------------------------------
BRAND   = "#6A2C91"   # main purple
ACCENT  = "#E08A1E"   # contrasting orange
SEG_COLORS = {
    "Champions": "#2E7D32", "Loyal": "#1976D2", "New/Promising": "#00ACC1",
    "Need Attention": "#F9A825", "At Risk": "#EF6C00", "Lost": "#C62828",
}

# --- Paths (deployment: configurable, never hard-coded) ----------------------
import os, sys
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "iv_retention").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
_candidates = [os.environ.get("IV_DATA_PATH", ""),
               PROJECT_ROOT / "data" / "Immaculate_Vibes_Sales.csv",
               Path("Immaculate_Vibes_Sales.csv"),
               PROJECT_ROOT / "sample_data" / "sample_sales.csv"]   # de-identified fallback
DATA_PATH = next(Path(p) for p in _candidates if p and Path(p).exists())
OUT_DIR = PROJECT_ROOT / "outputs"; OUT_DIR.mkdir(exist_ok=True)
print("Using data file:", DATA_PATH)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

def save_fig(name):
    """Tidy the layout, save the chart as a PNG and show it."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## 2. Data Preparation

Good analysis starts with knowing exactly what is in the data. In this section we load the file, check its quality, separate the orders that didn't result in revenue, and create a few helper columns the later modules rely on.

### 2.1 Load and inspect

We read the CSV and tell pandas that `order_datetime` is a real date-and-time (not just text), so we can do date maths later. We treat only truly **empty** cells as missing, so that no genuine text value is accidentally turned into a blank.

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["order_datetime"],
    keep_default_na=False,
    na_values=[""],
)

print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head(3).T   # transposed so all 47 columns are readable down the page

### 2.2 Quality checks

Before trusting any numbers we run the basic health checks: Is every `order_id` unique? What period does the data cover? How many customers are there? Does the order sequence number (`order_number_for_customer`) agree with the actual order dates?

In [ ]:
first_dt, last_dt = df["order_datetime"].min(), df["order_datetime"].max()

checks = pd.Series({
    "Rows (order lines)":              f"{len(df):,}",
    "Unique order_ids":                f"{df['order_id'].nunique():,}",
    "Duplicate order_ids":             int(df["order_id"].duplicated().sum()),
    "Unique customers":                f"{df['customer_id'].nunique():,}",
    "First order":                     first_dt.strftime("%Y-%m-%d"),
    "Last order":                      last_dt.strftime("%Y-%m-%d"),
    "Span (months)":                   round((last_dt - first_dt).days / 30.44, 1),
}, name="Value")
display(checks.to_frame())

# Does the supplied order sequence match the real chronological order?
seq_check = (df.sort_values(["customer_id", "order_datetime"])
               .groupby("customer_id").cumcount() + 1)
match_rate = (seq_check.reindex(df.index) == df["order_number_for_customer"]).mean()
print(f"order_number_for_customer agrees with the actual date order on {match_rate:.1%} of rows")

# Each customer should have one fixed county / age group
print("Customers with >1 county:   ", int((df.groupby('customer_id')['customer_county'].nunique() > 1).sum()))
print("Customers with >1 age group:", int((df.groupby('customer_id')['customer_age_group'].nunique() > 1).sum()))

### 2.3 Missing values

Next, which columns have blanks, and **are those blanks errors or expected?** A blank is only a problem if it *shouldn't* be blank. The two cross-tabs underneath check the reason for each gap.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame("missing_cells")
missing["pct_of_rows"] = missing["missing_cells"] / len(df) * 100
display(missing)

print("\nsleeve_type blanks by product type (blank = product has no sleeves):")
display(pd.crosstab(df["product_type"], df["sleeve_type"].fillna("(blank)")))

print("\nShare of rows with blank production_days / delivery_date, by order status:")
display(df.groupby("order_status")[["production_days", "delivery_date"]]
          .apply(lambda g: g.isna().mean()))

### 2.4 Revenue sanity check: which orders actually count?

Cancelled and Returned orders still show an `order_total` (what the customer *would* have paid), but no money was kept. If we add them into revenue, we'd overstate sales. Here we confirm that `net_revenue` is already zero for those orders, and we measure how much potential revenue was lost to them.

> `net_revenue` = the product value of a delivered order (it **excludes the delivery fee**), while `amount_collected` includes delivery. We use `net_revenue` for sales performance.

In [ ]:
status_summary = df.groupby("order_status").agg(
    orders=("order_id", "nunique"),
    order_total_value=("order_total", "sum"),
    amount_collected=("amount_collected", "sum"),
    net_revenue=("net_revenue", "sum"),
)
status_summary["share_of_orders"] = status_summary["orders"] / status_summary["orders"].sum()
display(status_summary)

# The key assumption of every revenue calculation below:
assert (df.loc[df["order_status"] != "Delivered", "net_revenue"] == 0).all()
print("Confirmed: net_revenue is 0 for every Cancelled / Returned order.")

lost = df.loc[df["order_status"] != "Delivered", "order_total"].sum()
print(f"\nOrder value lost to cancellations & returns: KES {lost:,.0f} "
      f"({lost / df['order_total'].sum():.1%} of all order value placed)")

### 2.5 Split the data: delivered vs cancelled/returned

We keep two views of the data:

* **`delivered`** – completed orders. Used for anything involving revenue (RFM monetary value, CLV, channel revenue…).
* **`df`** (all orders) – used for *behaviour* questions such as "when do people place orders?" and "who comes back?", because a customer who orders and then cancels is still showing interest.

Cancelled/returned orders are analysed on their own right below, not thrown away, because they are a finding in themselves.

In [ ]:
delivered     = df[df["order_status"] == "Delivered"].copy()
not_delivered = df[df["order_status"] != "Delivered"].copy()
print(f"Delivered: {len(delivered):,} orders   |   Cancelled/Returned: {len(not_delivered):,} orders")

# Where do cancellations and returns concentrate?
dims = {"Sales channel": "sales_channel", "Product type": "product_type", "Payment plan": "payment_plan"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), sharey=True)
for ax, (title, col) in zip(axes, dims.items()):
    rates = (pd.crosstab(df[col], df["order_status"], normalize="index")
               .reindex(columns=["Cancelled", "Returned"], fill_value=0)
               .sort_values("Cancelled", ascending=False))
    rates.plot(kind="bar", stacked=True, ax=ax, color=[ACCENT, BRAND], width=0.7)
    ax.set_title(f"Cancel / return rate by {title.lower()}")
    ax.set_xlabel(""); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("% of orders")
save_fig("02_cancellations_returns")

> ### What the data preparation told us
> * **The data is clean.** Every `order_id` is unique, the supplied order sequence matches the real order dates on 100% of rows, and each customer has exactly one county and age group.
> * **The blanks are expected, not errors.** `sleeve_type` is blank for every Palazzo pants order (no sleeves) and for some Kitenge sets. `production_days` and `delivery_date` are blank only for *Cancelled* orders, since nothing was produced or delivered. (`hip_size_inches` has no blanks, so the blueprint's note about it doesn't apply.)
> * **The time span is about 24 months** (Nov 2024 to Oct 2026), not the 2 months in the blueprint. That's good news: we can see seasonality and long-term loyalty, which a 2-month window couldn't show.
> * **185 of 2,115 orders (8.7%) produced no revenue**: 147 cancelled (6.9%) and 38 returned (1.8%). That's KES 576,550, or 8.0% of all order value placed.
> * **Cancellations aren't a TikTok problem.** Every channel sits between 6.3% and 7.3%, with TikTok and Referral tied at the top. Product matters more: Palazzo pants are cancelled most (8.5%) and standard 2-piece sets least (4.6%). Orders paid in full upfront are cancelled slightly more often than deposit + balance orders (about 8% vs 6%).
> * **31 customers never had a delivered order**, so they're left out of the RFM and CLV analysis (736 customers scored).

### 2.6 Derived fields

Now we add helper columns used throughout:

* `is_repeat` – is this order the customer's 2nd (or later)?
* `order_value_band` – groups basket size into Low / Medium / High / Premium.
* `time_band` – groups the hour of purchase into parts of the day.

In [ ]:
# Is this a repeat order (2nd, 3rd, ... purchase)?
df["is_repeat"] = df["order_number_for_customer"] > 1

# Order value bands (the last band is open-ended so the biggest baskets aren't dropped)
df["order_value_band"] = pd.cut(
    df["order_total"],
    bins=[0, 2000, 3500, 5000, np.inf],
    labels=["Low (≤2k)", "Medium (2k–3.5k)", "High (3.5k–5k)", "Premium (5k+)"],
)

# Part of the day
df["time_band"] = pd.cut(
    df["hour"], bins=[-1, 5, 11, 16, 20, 23],
    labels=["Late night (0–5)", "Morning (6–11)", "Afternoon (12–16)",
            "Evening (17–20)", "Night (21–23)"],
)

# Re-create the two subsets so they carry the new columns
delivered     = df[df["order_status"] == "Delivered"].copy()
not_delivered = df[df["order_status"] != "Delivered"].copy()

print(df["order_value_band"].value_counts().sort_index().to_frame("orders"))

### 2.7 One row per customer

Most questions in this project are about *customers*, not orders. So we build a customer table with each person's order count, first and last order date, lifetime revenue, and (importantly) the channel they **first** came in through, their **acquisition channel**. We also record when they placed their second order, which later lets us measure "how quickly do people come back?".

In [ ]:
all_sorted = df.sort_values(["customer_id", "order_datetime"])

customers = all_sorted.groupby("customer_id").agg(
    orders_placed=("order_id", "nunique"),
    first_order=("order_datetime", "min"),
    last_order=("order_datetime", "max"),
    first_channel=("sales_channel", "first"),      # acquisition channel
    gender=("customer_gender", "first"),
    age_group=("customer_age_group", "first"),
    county=("customer_county", "first"),
)

# Second order date (blank for one-time customers)
customers["second_order"] = all_sorted.groupby("customer_id")["order_datetime"].apply(
    lambda s: s.iloc[1] if len(s) > 1 else pd.NaT)
customers["days_to_second_order"] = (customers["second_order"] - customers["first_order"]).dt.days

# Revenue only counts delivered orders
rev = delivered.groupby("customer_id").agg(
    delivered_orders=("order_id", "nunique"),
    total_revenue=("net_revenue", "sum"),
)
customers = customers.join(rev).fillna({"delivered_orders": 0, "total_revenue": 0})

customers["is_repeat_customer"] = customers["orders_placed"] > 1
customers["repeat_within_180d"] = customers["days_to_second_order"] <= 180   # NaN → False

n_no_delivery = int((customers["delivered_orders"] == 0).sum())
print(f"Customers: {len(customers):,}  |  customers with no delivered order at all: {n_no_delivery}")
customers.head()

---
## 3. Module 1 – Customer Base Overview

**Goal:** set the baseline. How many customers do we have, and how many of them actually buy again?

### 3.1 Headline numbers

We count unique customers, then split them into *one-time* buyers (1 order) and *repeat* buyers (2+ orders). Note the difference between two measures that look similar:

* `customer_type` in the data is attached to each **order** ("was this order from a new or returning customer?"). Every customer has exactly one "New" order, their first.
* **Repeat rate** is a **customer** measure: what share of customers ever placed a second order?

In [ ]:
n_customers   = len(customers)
n_orders      = df["order_id"].nunique()
n_repeat      = int(customers["is_repeat_customer"].sum())
repeat_rate   = n_repeat / n_customers

kpis = pd.Series({
    "Total unique customers": f"{n_customers:,}",
    "Total orders placed": f"{n_orders:,}",
    "One-time customers": f"{n_customers - n_repeat:,}  ({1 - repeat_rate:.1%})",
    "Repeat customers (2+ orders)": f"{n_repeat:,}  ({repeat_rate:.1%})",
    "Average orders per customer": f"{customers['orders_placed'].mean():.2f}",
    "Median orders per customer": f"{customers['orders_placed'].median():.0f}",
    "Most orders by one customer": int(customers['orders_placed'].max()),
}, name="Value")
display(kpis.to_frame())

print("\nOrder-level split (customer_type):")
display(df["customer_type"].value_counts().to_frame("orders"))

### 3.2 How do the orders break down?

Three views of the same story: the left chart counts *orders* from new vs returning customers, the middle shows how many orders each customer has placed, and the right shows the share of one-time vs repeat *customers*.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# 1) New vs returning ORDERS
ct = df["customer_type"].value_counts().reindex(["New", "Returning"])
bars = axes[0].bar(ct.index, ct.values, color=[ACCENT, BRAND], width=0.55)
axes[0].bar_label(bars, fmt="{:,.0f}", padding=3)
axes[0].set_title("Orders from new vs returning customers")
axes[0].set_ylabel("Orders")

# 2) Orders per customer
max_o = customers["orders_placed"].max()
axes[1].hist(customers["orders_placed"], bins=np.arange(0.5, max_o + 1.5, 1),
             color=BRAND, edgecolor="white")
axes[1].set_title("How many orders has each customer placed?")
axes[1].set_xlabel("Orders per customer"); axes[1].set_ylabel("Customers")
axes[1].set_xticks(range(1, max_o + 1, 1 if max_o <= 12 else 2))

# 3) One-time vs repeat CUSTOMERS
split = customers["is_repeat_customer"].map({False: "One-time", True: "Repeat"}).value_counts()
axes[2].pie(split.reindex(["One-time", "Repeat"]), labels=["One-time", "Repeat"],
            colors=[ACCENT, BRAND], autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", width=0.55))
axes[2].set_title("Customers: one-time vs repeat")
save_fig("03_customer_base_overview")

### 3.3 The loyalty ladder: how many customers reach the 2nd, 3rd, 4th order?

This is one of the most useful retention views. For each order number we count how many customers ever reached it, and what share of them went on to place the *next* order (the "continue rate"). If the continue rate climbs with each order, loyalty is *self-reinforcing*: the more someone buys, the likelier they are to buy again.

In [ ]:
reach = df.groupby("order_number_for_customer")["customer_id"].nunique()
ladder = pd.DataFrame({"customers_reaching": reach})
ladder["pct_of_all_customers"] = ladder["customers_reaching"] / n_customers
ladder["continue_rate_to_next"] = ladder["customers_reaching"].shift(-1) / ladder["customers_reaching"]
display(ladder.head(10).style.format({"customers_reaching": "{:,.0f}",
                                       "pct_of_all_customers": "{:.1%}",
                                       "continue_rate_to_next": "{:.1%}"}))

top = ladder.head(10)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(top.index, top["customers_reaching"], color=BRAND, width=0.65)
for x, (n, c) in zip(top.index, zip(top["customers_reaching"], top["continue_rate_to_next"])):
    if pd.notna(c):
        ax.text(x, n + 8, f"{c:.0%}\ncontinue", ha="center", fontsize=8, color="#444")
ax.set_xticks(top.index); ax.set_xlabel("Order number"); ax.set_ylabel("Customers who reached it")
ax.set_title("The loyalty ladder (labels = share who go on to place the next order)")
ax.set_ylim(0, top["customers_reaching"].max() * 1.15)
save_fig("03_loyalty_ladder")

### 3.4 Our most loyal customers

A quick look at the heaviest buyers, in order count and in revenue.

In [ ]:
cols = ["orders_placed", "delivered_orders", "total_revenue", "first_channel", "county", "age_group", "gender"]
display(customers.sort_values(["orders_placed", "total_revenue"], ascending=False)[cols].head(10))

### 3.5 Is the business growing through new customers or returning ones?

Over two years, total orders have risen, but *who* is placing them? We split every month's orders into **first orders** (new customers) and **repeat orders** (returning customers), and then run a like-for-like comparison of January–October 2026 against January–October 2025.

In [ ]:
month = df["order_datetime"].dt.to_period("M").dt.to_timestamp()
mix = df.groupby([month, "customer_type"]).size().unstack(fill_value=0)[["New", "Returning"]]

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(mix.index, mix["Returning"], width=22, color=BRAND, label="Orders from returning customers")
ax.bar(mix.index, mix["New"], bottom=mix["Returning"], width=22, color=ACCENT, label="First orders (new customers)")
ax.set_title("Monthly orders: new vs returning customers")
ax.set_ylabel("Orders"); ax.legend(loc="upper left")
save_fig("03_new_vs_returning_over_time")

# Like-for-like: January-October of each year
ytd = {}
for y in (2025, 2026):
    d = df[(df["order_datetime"].dt.year == y) & (df["order_datetime"].dt.month <= 10)]
    ytd[y] = {"Total orders": len(d),
              "New customers": int((d["customer_type"] == "New").sum()),
              "Orders from returning customers": int((d["customer_type"] == "Returning").sum())}
ytd = pd.DataFrame(ytd).T
ytd.loc["Change 2026 vs 2025"] = ytd.loc[2026] / ytd.loc[2025] - 1
display(ytd.style.format("{:,.0f}", subset=pd.IndexSlice[[2025, 2026], :])
               .format("{:+.0%}", subset=pd.IndexSlice[["Change 2026 vs 2025"], :]))

> ### What we found: customer base
> * **This is a business of regulars.** Of 767 customers, **475 (61.9%) have ordered more than once** and only 292 bought a single time. The average customer places 2.8 orders, and the most loyal has placed 16. The blueprint's expectation of a 5–8% repeat rate came from a quick scan of a 2-month window; over two years the picture is very different.
> * **Loyalty reinforces itself, but only after a point.** Roughly 62–64% of customers go on to the next order at each of the first four steps. From the 5th order onwards the continue rate climbs to roughly 66–79%, so customers who reach five orders are the stickiest. (Beyond the 10th order, fewer than 25 customers remain, so those percentages are noise.)
> * **Our best customers look alike.** Of the top 10 customers by orders, 9 are women, 7 are aged 18–34, and 9 first came in through TikTok or Instagram. The top customer has 15 delivered orders.
> * **Growth is coming from existing customers, not new ones.** Comparing January–October 2026 with the same months of 2025, total orders are up 33%, orders from returning customers are up 108%, but first orders (new customers) are down 49%. The business is growing on the back of loyalty, which is great, but it also means the pipeline of new customers has thinned.

---
## 4. Module 2 – RFM Segmentation

**Goal:** score every customer on three simple questions and group them into actionable segments.

* **Recency** – how many days since their last purchase? (fewer = better)
* **Frequency** – how many delivered orders have they made? (more = better)
* **Monetary** – how much revenue have they brought in? (more = better)

### 4.1 Compute R, F and M for each customer

We use **delivered** orders only, because that's real money. The "snapshot date" is the day after the last order in the data, so "recency" is measured as of the end of the dataset, like running the report on the last day.

In [ ]:
snapshot_date = df["order_datetime"].max().normalize() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date.date())

rfm = delivered.groupby("customer_id").agg(
    Recency=("order_datetime", lambda x: (snapshot_date - x.max()).days),
    Frequency=("order_id", "nunique"),
    Monetary=("net_revenue", "sum"),
).reset_index()

print(f"Customers scored: {len(rfm):,}  (customers with no delivered order are excluded)")
rfm[["Recency", "Frequency", "Monetary"]].describe().T

### 4.2 Turn the raw numbers into 1–5 scores

We rank customers and cut them into five equal-sized groups (quintiles). **5 is always the best**, so a recent buyer gets R=5 and a long-lapsed one gets R=1.

One practical detail: many customers share the same Frequency (e.g. lots of people with exactly 2 orders). If we cut on raw Frequency, the groups wouldn't be equal. To keep the cuts fair, we rank customers with ties broken by their Monetary value, so a 2-order customer who spent more ranks above a 2-order customer who spent less.

In [ ]:
# Rank first (ties broken sensibly) so every quintile has an equal number of customers
rfm["R_score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)

freq_rank = (rfm["Frequency"] + rfm["Monetary"] / (rfm["Monetary"].max() * 10)).rank(method="first")
rfm["F_score"] = pd.qcut(freq_rank, 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["M_score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
rfm.head()

### 4.3 Name the segments

Scores are hard to act on, so we convert them into plain-English groups using simple rules:

| Segment | Rule | What it means |
|---|---|---|
| **Champions** | Recent *and* frequent (R≥4, F≥4) | Our best, most engaged customers |
| **At Risk** | Haven't bought lately but used to buy often (R≤2, F≥3) | Valuable people drifting away, **win back first** |
| **Loyal** | Frequent and high-spending (F≥3, M≥3) | Reliable repeat buyers who are still active |
| **New/Promising** | Recent but few orders (R≥4, F≤2) | Fresh customers who could become loyal |
| **Lost** | Not recent, few orders (R≤2, F≤2) | Probably gone |
| **Need Attention** | Everyone in between | Average on all fronts |

The rules are checked top to bottom, and a customer lands in the first one that fits.

> **A deliberate tweak to the original blueprint.** The blueprint checked "Loyal" *before* "At Risk". On this data that backfires: the Loyal rule doesn't look at recency, so it swallows lapsed high-value customers before the At Risk rule can see them (we saw "Loyal" customers who hadn't ordered in over a year). Checking **At Risk first** keeps the Loyal group genuinely active and gives the win-back group a real size.

In [ ]:
def assign_segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4:   return "Champions"
    elif row["R_score"] <= 2 and row["F_score"] >= 3: return "At Risk"      # checked BEFORE Loyal (see note above)
    elif row["F_score"] >= 3 and row["M_score"] >= 3: return "Loyal"
    elif row["R_score"] >= 4 and row["F_score"] <= 2: return "New/Promising"
    elif row["R_score"] <= 2 and row["F_score"] <= 2: return "Lost"
    else:                                              return "Need Attention"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)

SEG_ORDER = ["Champions", "Loyal", "New/Promising", "Need Attention", "At Risk", "Lost"]
seg = (rfm.groupby("Segment").agg(
            customers=("customer_id", "count"),
            avg_recency_days=("Recency", "mean"),
            avg_orders=("Frequency", "mean"),
            avg_revenue=("Monetary", "mean"),
            total_revenue=("Monetary", "sum"))
         .reindex(SEG_ORDER).fillna(0))
seg["customer_share"] = seg["customers"] / seg["customers"].sum()
seg["revenue_share"]  = seg["total_revenue"] / seg["total_revenue"].sum()

display(seg.style.format({"customers": "{:,.0f}", "avg_recency_days": "{:,.0f}", "avg_orders": "{:.1f}",
                           "avg_revenue": "{:,.0f}", "total_revenue": "{:,.0f}",
                           "customer_share": "{:.1%}", "revenue_share": "{:.1%}"}))

# How concentrated is revenue? (the classic "80/20" check)
rev_sorted = rfm["Monetary"].sort_values(ascending=False)
top20_share = rev_sorted.head(int(len(rev_sorted) * 0.2)).sum() / rev_sorted.sum()
print(f"\nThe top 20% of customers generate {top20_share:.1%} of delivered revenue.")

### 4.4 See the segments

Three charts: (1) each customer plotted by how recently they bought vs how much they've spent, with dot size showing order count; (2) how many customers sit in each segment; (3) how much revenue each segment contributes. Comparing (2) and (3) shows which segments punch above their weight.

In [ ]:
fig = plt.figure(figsize=(17, 5.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1, 1])

# 1) RFM scatter
ax = fig.add_subplot(gs[0])
for s in SEG_ORDER:
    d = rfm[rfm["Segment"] == s]
    ax.scatter(d["Recency"], d["Monetary"], s=20 + d["Frequency"] * 14,
               color=SEG_COLORS[s], alpha=0.55, label=s, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Recency (days since last order, lower = more recent)")
ax.set_ylabel("Lifetime revenue (KES)")
ax.set_title("RFM map: bigger dot = more orders")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
ax.legend(title="Segment", fontsize=8, title_fontsize=9, loc="upper right")

# 2) Customers per segment
ax = fig.add_subplot(gs[1])
bars = ax.bar(seg.index, seg["customers"], color=[SEG_COLORS[s] for s in seg.index])
ax.bar_label(bars, fmt="{:,.0f}", padding=2, fontsize=8)
ax.set_title("Customers per segment"); ax.tick_params(axis="x", rotation=40)

# 3) Revenue share per segment
ax = fig.add_subplot(gs[2])
bars = ax.bar(seg.index, seg["revenue_share"], color=[SEG_COLORS[s] for s in seg.index])
ax.bar_label(bars, labels=[f"{v:.0%}" for v in seg["revenue_share"]], padding=2, fontsize=8)
ax.set_title("Share of revenue per segment"); ax.tick_params(axis="x", rotation=40)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
save_fig("04_rfm_segments")

### 4.5 Save the segment table

The customer-level RFM table (with each customer's channel, county, age group and gender attached) is saved to `rfm_segments.csv`, ready to use for targeted campaigns, for example messaging every *At Risk* customer.

In [ ]:
rfm_out = rfm.merge(
    customers[["first_channel", "county", "age_group", "gender", "orders_placed"]].reset_index(),
    on="customer_id", how="left")
rfm_out.to_csv(OUT_DIR / "rfm_segments.csv", index=False)
print(f"Saved rfm_segments.csv  ({len(rfm_out):,} customers)")
rfm_out.sort_values("Monetary", ascending=False).head(8)

> ### What we found: RFM segments
> * **Two segments carry the business.** *Champions* are 24% of scored customers but bring in **46% of revenue** (average KES 16,467 each, about 5 orders). Add *Loyal* (20% of customers, 23% of revenue) and just under half the customer base generates **69% of revenue**. Overall, the top 20% of customers produce 49%.
> * **The biggest recoverable pool is "At Risk".** These are 107 customers (14%) who averaged 2.7 orders and KES 8,457 each, but haven't ordered for about 354 days on average. Together they've brought in KES 904,900 (15% of revenue). Winning even a fraction back is worth more than any other single segment.
> * **"Lost" customers are almost all one-and-done.** 187 customers (25%) bought exactly once and have been quiet for about 472 days. They account for 10% of revenue, but because they never showed repeat behaviour they're a lower win-back priority than At Risk.
> * **The "New/Promising" window is short.** 62 customers bought once recently (about 52 days ago on average). Whether they place a second order is the single biggest lever on future loyalty (see Module 6).

---
## 5. Module 3 – Channel Effectiveness and Retention

**Goal:** which channels produce loyal, high-value customers, and which only produce one-off buyers?

**An important subtlety.** Customers in this data don't stick to one channel: many buy through two, three or even all five channels over time. So "channel" can mean two different things, and we look at both:

1. **Order channel**: the channel a given *order* came through (good for volume, revenue, order value and cancellations).
2. **Acquisition channel**: the channel a customer's *first* order came through (the fair way to ask "which channel brings in loyal customers?").

### 5.1 Build the channel scorecard

In [ ]:
# --- Order channel view (what happens in each channel) -----------------------
by_order_ch = df.groupby("sales_channel").agg(
    orders=("order_id", "nunique"),
    customers_ordering=("customer_id", "nunique"),
    returning_order_share=("is_repeat", "mean"),
    cancellation_rate=("order_status", lambda s: (s == "Cancelled").mean()),
    return_rate=("order_status", lambda s: (s == "Returned").mean()),
)
rev_ch = delivered.groupby("sales_channel").agg(
    revenue=("net_revenue", "sum"),
    avg_order_value=("net_revenue", "mean"),
)

# --- Acquisition channel view (who brings in loyal customers) -----------------
acq = customers.groupby("first_channel").agg(
    customers_acquired=("orders_placed", "size"),
    repeat_rate=("is_repeat_customer", "mean"),
    avg_orders_per_customer=("orders_placed", "mean"),
    avg_lifetime_revenue=("total_revenue", "mean"),
)

# Fairness fix: customers acquired recently haven't had time to come back yet.
# So we also measure "came back within 180 days" using only customers acquired >180 days before the data ends.
cutoff = df["order_datetime"].max() - pd.Timedelta(days=180)
mature = customers[customers["first_order"] <= cutoff]
acq["repeat_within_180d"] = mature.groupby("first_channel")["repeat_within_180d"].mean()

scorecard = by_order_ch.join(rev_ch).join(acq).sort_values("revenue", ascending=False)
scorecard["revenue_share"] = scorecard["revenue"] / scorecard["revenue"].sum()

fmt = {"orders": "{:,.0f}", "customers_ordering": "{:,.0f}", "returning_order_share": "{:.1%}",
       "cancellation_rate": "{:.1%}", "return_rate": "{:.1%}", "revenue": "{:,.0f}",
       "avg_order_value": "{:,.0f}", "customers_acquired": "{:,.0f}", "repeat_rate": "{:.1%}",
       "avg_orders_per_customer": "{:.2f}", "avg_lifetime_revenue": "{:,.0f}",
       "repeat_within_180d": "{:.1%}", "revenue_share": "{:.1%}"}
print(f"(180-day measure uses {len(mature):,} customers acquired before {cutoff.date()})")
display(scorecard.style.format(fmt))

### 5.2 Visualise the channels

* **Left:** new vs returning orders per channel (does a channel mostly serve first-timers or regulars?)
* **Middle:** average order value vs repeat rate for each *acquisition* channel; the top-right corner is the ideal (high-value *and* loyal). Bubble size = customers acquired.
* **Right:** a heatmap of revenue by channel and customer type.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Grouped bar: new vs returning orders by channel
nr = (pd.crosstab(df["sales_channel"], df["customer_type"])
        .reindex(scorecard.index)[["New", "Returning"]])
nr.plot(kind="bar", ax=axes[0], color=[ACCENT, BRAND], width=0.75)
axes[0].set_title("New vs returning orders by channel")
axes[0].set_xlabel(""); axes[0].set_ylabel("Orders"); axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="Customer type")

# 2) Scatter: AOV vs repeat rate (acquisition channel)
sc = scorecard
axes[1].scatter(sc["avg_order_value"], sc["repeat_rate"], s=sc["customers_acquired"] * 2.2,
                color=BRAND, alpha=0.6, edgecolor="white")
for ch, r in sc.iterrows():
    axes[1].annotate(ch, (r["avg_order_value"], r["repeat_rate"]), ha="center", va="center", fontsize=9, color="black")
axes[1].set_xlabel("Average order value (KES, delivered orders)")
axes[1].set_ylabel("Repeat rate of customers acquired")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Value vs loyalty (bubble = customers acquired)")
axes[1].margins(0.18)

# 3) Heatmap: channel x customer type revenue
heat = delivered.pivot_table(index="sales_channel", columns="customer_type",
                             values="net_revenue", aggfunc="sum").reindex(scorecard.index)
sns.heatmap(heat, annot=True, fmt=",.0f", cmap="Purples", cbar=False, ax=axes[2], linewidths=0.5)
axes[2].set_title("Delivered revenue (KES): channel × customer type")
axes[2].set_xlabel(""); axes[2].set_ylabel("")
save_fig("05_channel_effectiveness")

### 5.3 Do customers switch channels?

A quick check on how common channel-hopping is. If most customers use several channels, then a "TikTok customer" is really a *customer who started on TikTok*, which is why we use acquisition channel for loyalty comparisons.

In [ ]:
channels_used = df.groupby("customer_id")["sales_channel"].nunique()
hop = channels_used.value_counts().sort_index().to_frame("customers")
hop["share"] = hop["customers"] / hop["customers"].sum()
hop.index.name = "channels_used"
display(hop.style.format({"share": "{:.1%}"}))

# Among repeat customers only (one-time buyers can't hop)
rep_ids = customers.index[customers["is_repeat_customer"]]
print(f"Repeat customers who used 2+ channels: {(channels_used[rep_ids] > 1).mean():.1%}")

> ### What we found: channels
> * **TikTok is the volume engine.** It acquired 355 of the 767 customers (46%) and produced 42% of delivered revenue (KES 2.60M). WhatsApp is second with 28% of revenue.
> * **Loyalty differences are modest, and Instagram quietly leads.** Customers acquired through Instagram place the most orders (3.1 each), are worth the most over their lifetime (KES 9,394), and are the most likely to reorder within 180 days (43%). Referral customers have the highest "ever came back" rate (65%) but the *lowest* 180-day rate (31%), so they do return, just more slowly. This contradicts the blueprint's guess that Referral would be the loyalty champion.
> * **Walk-in isn't a premium channel here.** It has the lowest average order value (KES 3,039) and the lowest lifetime revenue per customer (KES 5,905), though with only 31 customers acquired, treat that with caution.
> * **Order values are almost identical across channels** (KES 3,039–3,253). Channels differ in *who* they bring in and how often they return, not in basket size.
> * **Customers move between channels.** 53% of all customers, and **85% of repeat customers**, have used two or more channels. A "TikTok customer" is really someone who *started* on TikTok. That's why the retention follow-up should happen wherever the customer is (usually WhatsApp), not only in the channel that first won them.
> * **A word of caution:** Referral (75), Instagram (110) and Walk-in (31) are small groups, so differences of a few percentage points could be chance. The Instagram lead on orders per customer and lifetime revenue is the most consistent signal.

---
## 6. Module 4 – Demographic and Geographic Segmentation

**Goal:** who is the most valuable customer, in terms of age, gender and place?

All revenue figures here use **delivered** orders. We look at both the *total* (how much a group is worth to the business) and the *average* (how valuable each order or customer is), because a big group of small spenders can look very different from a small group of big spenders.

### 6.1 Age groups

In [ ]:
age = delivered.groupby("customer_age_group").agg(
    revenue=("net_revenue", "sum"),
    avg_order_value=("net_revenue", "mean"),
    orders=("order_id", "nunique"),
    customers=("customer_id", "nunique"),
)
age["revenue_per_customer"] = age["revenue"] / age["customers"]
age["revenue_share"] = age["revenue"] / age["revenue"].sum()
display(age.style.format({"revenue": "{:,.0f}", "avg_order_value": "{:,.0f}", "orders": "{:,.0f}",
                           "customers": "{:,.0f}", "revenue_per_customer": "{:,.0f}", "revenue_share": "{:.1%}"}))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (col, title, color) in zip(axes, [
        ("revenue", "Total revenue by age group", BRAND),
        ("avg_order_value", "Average order value by age group", ACCENT),
        ("revenue_per_customer", "Revenue per customer by age group", "#2E7D32")]):
    bars = ax.bar(age.index, age[col], color=color, width=0.6)
    ax.bar_label(bars, labels=[f"{v/1e6:.2f}M" if col == "revenue" else f"{v:,.0f}" for v in age[col]], padding=2, fontsize=8)
    ax.set_title(title); ax.set_xlabel("Age group")
    ax.set_ylim(0, age[col].max() * 1.15)
save_fig("06_age_groups")

### 6.2 Counties

Where does the revenue come from? Note that "Other" is a catch-all category in the data (all the counties too small to list individually), not a real place.

In [ ]:
county = delivered.groupby("customer_county").agg(
    revenue=("net_revenue", "sum"),
    avg_order_value=("net_revenue", "mean"),
    orders=("order_id", "nunique"),
    customers=("customer_id", "nunique"),
).sort_values("revenue", ascending=False)
county["revenue_share"] = county["revenue"] / county["revenue"].sum()
county["cumulative_share"] = county["revenue_share"].cumsum()
display(county.style.format({"revenue": "{:,.0f}", "avg_order_value": "{:,.0f}", "orders": "{:,.0f}",
                              "customers": "{:,.0f}", "revenue_share": "{:.1%}", "cumulative_share": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
c = county.sort_values("revenue")
axes[0].barh(c.index, c["revenue"], color=BRAND)
axes[0].set_title("Revenue by county (KES)")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
c2 = county.sort_values("avg_order_value")
axes[1].barh(c2.index, c2["avg_order_value"], color=ACCENT)
axes[1].set_title("Average order value by county (KES)")
save_fig("06_counties")

### 6.3 Gender and product preferences

Which products do different groups buy? Because women make up the vast majority of orders, raw counts would hide how men's buying actually differs. So for the charts we convert to **percentages within each group** (e.g. "of all products bought by men, what share were Kimono sets?").

In [ ]:
# Raw counts first
print("Order counts: gender × product type")
display(pd.crosstab(df["customer_gender"], df["product_type"], margins=True))

# Percentages within each group
gender_mix = pd.crosstab(df["customer_gender"], df["product_type"], normalize="index")
age_mix    = pd.crosstab(df["customer_age_group"], df["product_type"], normalize="index")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
gender_mix.plot(kind="bar", stacked=True, ax=axes[0], colormap="viridis", width=0.55)
axes[0].set_title("Product mix by gender (% of orders)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xlabel(""); axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Product", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

sns.heatmap(age_mix, annot=True, fmt=".0%", cmap="Purples", ax=axes[1], cbar=False, linewidths=0.5)
axes[1].set_title("Product preference by age group (% of orders)")
axes[1].set_xlabel(""); axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=20)
save_fig("06_product_preferences")

### 6.4 Who is the most valuable customer profile?

Finally we combine age group and gender into customer profiles and rank them by **revenue per customer**. We only show profiles with at least 15 customers so one big spender can't distort the picture.

In [ ]:
profile = (customers.groupby(["age_group", "gender"])
           .agg(customers=("orders_placed", "size"),
                avg_orders=("orders_placed", "mean"),
                revenue_per_customer=("total_revenue", "mean"),
                total_revenue=("total_revenue", "sum"))
           .query("customers >= 15")
           .sort_values("revenue_per_customer", ascending=False))
display(profile.style.format({"customers": "{:,.0f}", "avg_orders": "{:.2f}",
                               "revenue_per_customer": "{:,.0f}", "total_revenue": "{:,.0f}"}))

> ### What we found: demographics and geography
> * **Age:** 25–34 is the biggest revenue pool (KES 2.32M, 37%), but mostly because it's the biggest group (281 customers). Per customer, **18–24 is the most valuable** (KES 8,915 vs roughly 8,150–8,420 for the others). Average order value is flat across ages (KES 3,160–3,265), and 45+ is actually the lowest, so the idea that older customers spend more per order isn't supported by this data.
> * **Geography:** Nairobi dominates at **37% of revenue** (726 orders), and Nairobi plus Kiambu together make up about 43%, well short of the 60% the blueprint expected. Revenue is spread widely: Mombasa, Nakuru, Kisumu and Kiambu each contribute about 7%. Average order value is a little higher outside Nairobi (Meru KES 3,411, Kisumu 3,395, Machakos 3,376 vs Nairobi 3,138 and Kiambu 3,067), a gap of about 8–10% that's worth watching but based on small groups of 31–39 customers.
> * **Gender and products:** Men placed just 143 of 2,115 orders (6.8%). Product choices are similar across genders; men lean slightly more toward Kimono sets (19% vs 15%), but that's only 27 orders. Palazzo pants are the single biggest seller in almost every age group (31–33% of orders).
> * **The core customer** is a woman aged 18–34: women in those two age bands generate about **62% of all delivered revenue**. The profile with the highest revenue per customer is men aged 35–44 (KES 9,937), but that's only 15 customers.

---
## 7. Module 5 – Purchase Timing and Behavioural Patterns

**Goal:** when do customers buy, and does timing relate to basket size?

For *timing* questions we use **all placed orders** (not just delivered), since the moment someone chooses to order is what matters, even if the order was later cancelled. For basket size we use `order_total`, the value of what the customer put in their order.

### 7.1 When are orders placed? Weekday × hour heatmap

Each cell counts how many orders were placed in that weekday-and-hour slot over the whole period. Darker = busier. This is the chart to use when deciding when to post content or run ads.

In [ ]:
heat = (df.pivot_table(index="weekday", columns="hour", values="order_id", aggfunc="count", fill_value=0)
          .reindex(WEEKDAYS)
          .reindex(columns=range(24), fill_value=0))

fig, ax = plt.subplots(figsize=(15, 4.5))
sns.heatmap(heat, cmap="Purples", annot=True, fmt="d", annot_kws={"size": 7}, linewidths=0.4, cbar_kws={"label": "Orders"}, ax=ax)
ax.set_title("Orders by weekday and hour of day")
ax.set_xlabel("Hour of day (24h)"); ax.set_ylabel("")
save_fig("07_weekday_hour_heatmap")

# Orders per weekday, and per *occurrence* of that weekday (each weekday appears ~104 times in 24 months)
wd = df["weekday"].value_counts().reindex(WEEKDAYS)
day_counts = pd.Series(pd.date_range(df["order_datetime"].min().normalize(),
                                     df["order_datetime"].max().normalize()).day_name()).value_counts()
wd_table = wd.to_frame("orders")
wd_table["share"] = wd_table["orders"] / wd_table["orders"].sum()
wd_table["orders_per_that_day"] = wd_table["orders"] / day_counts.reindex(WEEKDAYS)
print("Orders by weekday:")
display(wd_table.style.format({"share": "{:.1%}", "orders_per_that_day": "{:.1f}"}))

### 7.2 Weekend vs weekday, and time of day

A fair weekend-vs-weekday comparison has to account for there being 5 weekdays but only 2 weekend days, so we compare **orders per calendar day**. Then we check basket size by part of the day and whether new and returning customers shop at different times.

In [ ]:
# --- Weekend vs weekday, per calendar day -----------------------------------
all_days = pd.date_range(df["order_datetime"].min().normalize(), df["order_datetime"].max().normalize())
n_weekend_days = int((all_days.dayofweek >= 5).sum())
n_weekday_days = len(all_days) - n_weekend_days

wk = df.groupby("is_weekend").agg(orders=("order_id", "count"),
                                  avg_order_total=("order_total", "mean"),
                                  median_order_total=("order_total", "median"))
wk["calendar_days"] = [n_weekday_days, n_weekend_days]   # False (weekday) first, then True (weekend)
wk["orders_per_day"] = wk["orders"] / wk["calendar_days"]
wk.index = ["Weekday", "Weekend"]
display(wk)

# --- Time of day -------------------------------------------------------------
tb = df.groupby("time_band", observed=True).agg(orders=("order_id", "count"),
                                                avg_order_total=("order_total", "mean"))
tb["share_of_orders"] = tb["orders"] / tb["orders"].sum()
display(tb.style.format({"orders": "{:,.0f}", "avg_order_total": "{:,.0f}", "share_of_orders": "{:.1%}"}))

# --- New vs returning timing ------------------------------------------------------
timing_mix = pd.crosstab(df["time_band"], df["customer_type"], normalize="columns")
print("\nShare of each customer type's orders placed in each part of the day:")
display(timing_mix.style.format("{:.1%}"))

### 7.3 Does basket size change by hour?

If late-night shoppers spend more (or less), that's useful for deciding which offers to push when. Dots are sized by the number of orders so thin hours don't mislead us.

In [ ]:
hourly = df.groupby("hour").agg(avg_order_total=("order_total", "mean"), orders=("order_id", "count"))

fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(hourly.index, hourly["avg_order_total"], color=BRAND, lw=1.5)
ax.scatter(hourly.index, hourly["avg_order_total"], s=hourly["orders"] * 1.2, color=BRAND, alpha=0.65, zorder=3)
ax.axhline(df["order_total"].mean(), ls="--", color="grey", lw=1, label="Overall average")
ax.set_xticks(range(0, 24)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Average order total (KES)")
ax.set_title("Average basket size by hour (dot size = number of orders)")
ax.legend()
save_fig("07_aov_by_hour")

### 7.4 How has demand moved over the 24 months?

Two views of the trend: a month-by-month line of orders and delivered revenue, then a **year-on-year** comparison (each year as its own line) to reveal seasonality, such as whether the festive season is reliably stronger.

In [ ]:
monthly = df.groupby(df["order_datetime"].dt.to_period("M").dt.to_timestamp()).agg(orders=("order_id", "count"))
monthly["delivered_revenue"] = delivered.groupby(delivered["order_datetime"].dt.to_period("M").dt.to_timestamp())["net_revenue"].sum()

fig, axes = plt.subplots(1, 2, figsize=(17, 4.6))
axes[0].plot(monthly.index, monthly["orders"], marker="o", color=BRAND, label="Orders")
axes[0].set_ylabel("Orders placed", color=BRAND)
ax2 = axes[0].twinx()
ax2.plot(monthly.index, monthly["delivered_revenue"] / 1000, marker="s", color=ACCENT, label="Delivered revenue")
ax2.set_ylabel("Delivered revenue (KES '000)", color=ACCENT); ax2.grid(False)
axes[0].set_title("Monthly orders and delivered revenue")
fig.autofmt_xdate()

yoy = (df.assign(year=df["order_datetime"].dt.year, month=df["order_datetime"].dt.month)
         .groupby(["month", "year"]).size().unstack("year"))
yoy.index = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
yoy.plot(ax=axes[1], marker="o", color=[ACCENT, BRAND, "#2E7D32"][:yoy.shape[1]])
axes[1].set_title("Orders per month, year on year"); axes[1].set_ylabel("Orders"); axes[1].set_xlabel("")
axes[1].legend(title="Year")
save_fig("07_monthly_trend")

> ### What we found: timing
> * **Orders arrive between 07:00 and 22:59, and the evening is the peak.** The 17:00–20:59 window brings **37% of orders**, with 18:00 and 19:00 the busiest hours. Afternoon and evening together account for 69%. Only 7% come after 21:00.
> * **Saturday is a clear peak and Sunday is a dead day.** Saturday gets about **4.6 orders per Saturday** (23% of all orders), against about 2.7–3.1 on a typical weekday and only **1.0 on a Sunday**. Weekend *as a whole* isn't stronger (2.80 vs 2.94 orders per day) because Sunday drags it down, so "weekend" is the wrong label; "Saturday" is the right one.
> * **Timing changes volume, not value.** Basket size is steady across the day (about KES 3,250–3,550 for nearly every hour) and across weekday vs weekend (KES 3,403 vs 3,466). The only outlier is 22:00 (KES 4,067), but that's just 42 orders. New and returning customers shop at the same times of day.
> * **There's clear seasonality and strong growth.** Demand dips from March to May (46–51 orders a month in 2025), builds from September, and peaks in December (156 orders in Dec 2025 vs 109 in Dec 2024). Orders in Jan–Oct 2026 are 33% above the same months of 2025, and Sep–Oct 2026 (123 and 131 orders) are the highest non-December months on record. **Expect a very busy Nov–Dec 2026**, so production capacity is worth planning for.

---
## 8. Module 6 – Customer Lifetime Value (CLV) and Retention Deep-Dive

**Goal:** put a price tag on loyalty, and find out who is slipping away.

### 8.1 What is a repeat customer worth compared to a one-time customer?

For each customer we total their delivered revenue, orders, average order value and the number of days between their first and last delivered order (their "lifespan"). We then compare repeat and one-time customers side by side. This is a **historical** CLV, i.e. what each customer has actually generated so far, not a forecast.

In [ ]:
clv = delivered.groupby("customer_id").agg(
    total_revenue=("net_revenue", "sum"),
    total_orders=("order_id", "nunique"),
    avg_order_value=("net_revenue", "mean"),
    first_delivered=("order_datetime", "min"),
    last_delivered=("order_datetime", "max"),
).reset_index()
clv["lifespan_days"] = (clv["last_delivered"] - clv["first_delivered"]).dt.days
clv["is_repeat"] = clv["total_orders"] > 1
clv["customer_type"] = clv["is_repeat"].map({False: "One-time", True: "Repeat"})

compare = clv.groupby("customer_type").agg(
    customers=("customer_id", "count"),
    avg_revenue=("total_revenue", "mean"),
    median_revenue=("total_revenue", "median"),
    avg_order_value=("avg_order_value", "mean"),
    total_revenue=("total_revenue", "sum"),
)
compare["revenue_share"] = compare["total_revenue"] / compare["total_revenue"].sum()
display(compare.style.format({"customers": "{:,.0f}", "avg_revenue": "{:,.0f}", "median_revenue": "{:,.0f}",
                               "avg_order_value": "{:,.0f}", "total_revenue": "{:,.0f}", "revenue_share": "{:.1%}"}))

ratio = compare.loc["Repeat", "avg_revenue"] / compare.loc["One-time", "avg_revenue"]
print(f"A repeat customer is worth {ratio:.1f}× a one-time customer on average "
      f"(KES {compare.loc['Repeat','avg_revenue']:,.0f} vs KES {compare.loc['One-time','avg_revenue']:,.0f}).")

### 8.2 Revenue distribution and how value grows with each order

* **Left:** a box plot of lifetime revenue for one-time vs repeat customers (the box is the middle 50% of customers, the line inside is the median).
* **Right:** the average value of the 1st, 2nd, 3rd... delivered order. Do loyal customers spend *more per order* as they come back, or just buy more often? (Orders beyond the 8th are grouped together because few customers reach them.)

In [ ]:
# Number each customer's delivered orders 1, 2, 3 ...
d_sorted = delivered.sort_values(["customer_id", "order_datetime"]).copy()
d_sorted["delivered_seq"] = d_sorted.groupby("customer_id").cumcount() + 1
d_sorted["days_since_prev"] = d_sorted.groupby("customer_id")["order_datetime"].diff().dt.days
d_sorted["seq_bucket"] = d_sorted["delivered_seq"].clip(upper=8).astype(str).replace({"8": "8+"})
seq_order = [str(i) for i in range(1, 8)] + ["8+"]

by_seq = d_sorted.groupby("seq_bucket").agg(avg_order_value=("net_revenue", "mean"),
                                            orders=("order_id", "count")).reindex(seq_order)
display(by_seq.style.format({"avg_order_value": "{:,.0f}", "orders": "{:,.0f}"}))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.boxplot(data=clv, x="customer_type", y="total_revenue", order=["One-time", "Repeat"],
            palette=[ACCENT, BRAND], ax=axes[0], width=0.5)
axes[0].set_title("Lifetime revenue: one-time vs repeat customers")
axes[0].set_xlabel(""); axes[0].set_ylabel("Lifetime delivered revenue (KES)")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))

bars = axes[1].bar(by_seq.index, by_seq["avg_order_value"], color=BRAND, width=0.6)
axes[1].bar_label(bars, fmt="{:,.0f}", padding=2, fontsize=8)
axes[1].set_title("Average order value by order number")
axes[1].set_xlabel("Delivered order number"); axes[1].set_ylabel("Average order value (KES)")
axes[1].set_ylim(0, by_seq["avg_order_value"].max() * 1.15)
save_fig("08_clv_comparison")

### 8.3 How long until a customer comes back?

For every repeat purchase we measure the number of days since the customer's previous delivered order. That gives us the typical "repurchase rhythm", which is the basis for deciding **when** a follow-up message should go out and **when** a customer should be called "lapsed".

In [ ]:
gaps = d_sorted.dropna(subset=["days_since_prev"]).copy()
gaps["transition"] = (gaps["delivered_seq"] - 1).astype(int).astype(str) + " → " + gaps["delivered_seq"].astype(int).astype(str)

print(f"Repeat purchases measured: {len(gaps):,}")
print("\nDays between consecutive orders (all transitions):")
display(gaps["days_since_prev"].describe(percentiles=[.25, .5, .75, .9]).to_frame("days"))

first_trans = (gaps[gaps["delivered_seq"] <= 6].groupby("transition")["days_since_prev"]
               .agg(["count", "median", "mean"]))
print("\nBy step (1→2, 2→3, ...):")
display(first_trans.style.format({"mean": "{:.0f}", "median": "{:.0f}"}))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(gaps["days_since_prev"], bins=40, color=BRAND, edgecolor="white")
med = gaps["days_since_prev"].median(); p90 = gaps["days_since_prev"].quantile(0.9)
axes[0].axvline(med, color=ACCENT, ls="--", lw=2, label=f"Median: {med:.0f} days")
axes[0].axvline(p90, color="#C62828", ls="--", lw=2, label=f"90th percentile: {p90:.0f} days")
axes[0].set_title("Days between consecutive orders"); axes[0].set_xlabel("Days"); axes[0].set_ylabel("Repeat purchases")
axes[0].legend()

bars = axes[1].bar(first_trans.index, first_trans["median"], color=BRAND, width=0.6)
axes[1].bar_label(bars, fmt="{:.0f}", padding=2, fontsize=8)
axes[1].set_title("Median days between orders, by step"); axes[1].set_ylabel("Days")
save_fig("08_repurchase_interval")

### 8.4 Which repeat customers are at risk of churning?

The blueprint suggests flagging anyone with "no order in the last 14 days", but that rule was designed for a 2-month dataset. With two years of data and a repurchase rhythm measured in months, 14 days would flag nearly everyone.

Instead we let the data set the thresholds, using the gaps we just measured:

* **Active**: quiet for no longer than 75% of all repurchase gaps (the 75th percentile).
* **Drifting** *(early warning)*: quiet for longer than that, but not yet extreme.
* **Lapsed**: quiet for longer than **90%** of all repurchase gaps, longer than almost anyone who *did* come back.

We then rank the Drifting and Lapsed customers by lifetime value so the win-back list starts with the people most worth saving.

In [ ]:
drift_threshold = gaps["days_since_prev"].quantile(0.75)
churn_threshold = gaps["days_since_prev"].quantile(0.90)
print(f"Drifting: silent for more than {drift_threshold:.0f} days   |   Lapsed: silent for more than {churn_threshold:.0f} days")

repeat_clv = clv[clv["is_repeat"]].copy()
repeat_clv["days_since_last_order"] = (snapshot_date - repeat_clv["last_delivered"]).dt.days
repeat_clv["status"] = pd.cut(repeat_clv["days_since_last_order"],
                              bins=[-1, drift_threshold, churn_threshold, np.inf],
                              labels=["Active", "Drifting", "Lapsed"])

status = repeat_clv.groupby("status", observed=True).agg(customers=("customer_id", "count"),
                                          avg_revenue=("total_revenue", "mean"),
                                          total_revenue=("total_revenue", "sum"))
status["share_of_repeat_customers"] = status["customers"] / status["customers"].sum()
display(status.style.format({"customers": "{:,.0f}", "avg_revenue": "{:,.0f}",
                              "total_revenue": "{:,.0f}", "share_of_repeat_customers": "{:.1%}"}))

winback = (repeat_clv[repeat_clv["status"].isin(["Drifting", "Lapsed"])]
           .merge(customers[["first_channel", "county", "age_group"]].reset_index(), on="customer_id")
           .merge(rfm[["customer_id", "Segment"]], on="customer_id", how="left")
           .sort_values("total_revenue", ascending=False)
           [["customer_id", "status", "total_orders", "total_revenue", "days_since_last_order", "Segment",
             "first_channel", "county", "age_group"]])
winback.to_csv(OUT_DIR / "winback_list_repeat_customers.csv", index=False)
print(f"\nSaved winback_list_repeat_customers.csv ({len(winback):,} customers). Top 10 by value:")
display(winback.head(10))

### 8.5 Retention cohorts

A **cohort** is a group of customers who made their first purchase in the same month. The table shows, for each cohort, what percentage placed another order 1, 2, 3... months later. Read across a row to see how one cohort behaves over time, and down a column to see whether newer cohorts retain better or worse than older ones. Blank cells are months that haven't happened yet. (We count any order placed, delivered or not, as the customer being "active".)

In [ ]:
co = df[["customer_id", "order_datetime"]].copy()
co["order_month"] = co["order_datetime"].dt.to_period("M")
co["cohort"] = co.groupby("customer_id")["order_month"].transform("min")
co["offset"] = ((co["order_month"].dt.year * 12 + co["order_month"].dt.month)
                - (co["cohort"].dt.year * 12 + co["cohort"].dt.month))

active = co.groupby(["cohort", "offset"])["customer_id"].nunique().unstack(fill_value=0)
cohort_size = active[0]
retention = active.div(cohort_size, axis=0)

# Blank out months that haven't happened yet (so they don't look like "0% retained")
last = df["order_datetime"].max()
last_idx = last.year * 12 + last.month
for c in retention.index:
    max_off = last_idx - (c.year * 12 + c.month)
    retention.loc[c, retention.columns > max_off] = np.nan

MAX_OFFSET = 12
shown = retention.loc[:, 1:MAX_OFFSET]
shown.index = [f"{c}  (n={cohort_size[c]})" for c in shown.index]

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(shown, annot=True, fmt=".0%", cmap="Purples", annot_kws={"size": 7.5},
            linewidths=0.4, cbar_kws={"label": "% of cohort ordering again"}, ax=ax, vmin=0)
ax.set_title("Cohort retention: % of each first-purchase-month cohort that ordered N months later")
ax.set_xlabel("Months since first purchase"); ax.set_ylabel("Cohort (first-order month, n = new customers)")
ax.grid(False)
save_fig("08_cohort_retention")

# Overall average by month offset (only counting cohorts old enough to have reached it)
avg_ret = {}
for k in range(1, MAX_OFFSET + 1):
    ok = [c for c in active.index if last_idx - (c.year * 12 + c.month) >= k]
    avg_ret[k] = active.loc[ok, k].sum() / cohort_size[ok].sum()
print("Average % of a cohort that is active N months after first purchase:")
display(pd.Series(avg_ret, name="avg_retention").to_frame().T.style.format("{:.1%}"))

# --- How fast do first-time customers place a second order? (cumulative) --------
rows = []
for d in [30, 90, 180, 365]:
    eligible = customers[customers["first_order"] <= df["order_datetime"].max() - pd.Timedelta(days=d)]
    rows.append({"within_days": d, "customers_measured": len(eligible),
                 "placed_2nd_order": (eligible["days_to_second_order"] <= d).mean()})
conv = pd.DataFrame(rows).set_index("within_days")
print("\nShare of customers who placed a 2nd order within N days of their first (only customers old enough to be measured):")
display(conv.style.format({"customers_measured": "{:,.0f}", "placed_2nd_order": "{:.1%}"}))

> ### What we found: lifetime value and retention
> * **A repeat customer is worth 3.9× a one-time customer** (KES 11,970 vs KES 3,108 on average; medians KES 9,475 vs 3,100), and repeat customers generate **85% of all revenue**. (These counts, 442 repeat and 294 one-time, differ slightly from Module 1 because here "repeat" means two or more *delivered* orders.)
> * **The extra value comes from frequency, not bigger baskets.** Average order value is about the same for repeat (KES 3,231) and one-time customers (KES 3,108), and it doesn't rise with each successive order. Loyal customers don't "trade up"; they simply come back. Retention is therefore the way to grow lifetime value.
> * **The second order is the hardest, and it gets easier after that.** The median gap between orders is 98 days overall, but the 1→2 gap is longest (median 157 days) and then shortens: 119, 75, 84, 52 days for the later steps. Once the habit forms, customers return faster. (This only measures customers who did come back.)
> * **Most "one-timers" aren't lost yet.** Only 10% of customers place a second order within 30 days and 24% within 90, but 37% do within 180 days and **62% within a year**. So a lot of retention happens in months 4–12, and a customer acquired a few months ago who hasn't reordered is not a lost cause.
> * **Cohorts look steady, not improving.** In any given later month, roughly 6–13% of a cohort places an order. The Nov–Dec 2024 cohorts are much bigger (100 and 80 new customers) than most later ones (between 9 and 54 in nearly every month), which matches the slowdown in new customers seen in Module 1.
> * **The win-back list has 141 repeat customers worth KES 1.27M of past revenue.** 101 are *Drifting* (silent for over 209 days, KES 953,650 of past revenue) and 40 are *Lapsed* (over 351 days, KES 312,900). The most valuable are at the top of the saved list.

---
## 9. Wrap-up

### 9.1 Executive summary (computed live from the analysis above)

This cell pulls the headline figures directly from the results, so the numbers always match the data if you re-run the notebook.

In [ ]:
best_loyalty_ch   = scorecard["repeat_within_180d"].idxmax()
best_aov_ch       = scorecard["avg_order_value"].idxmax()
worst_cancel_ch   = scorecard["cancellation_rate"].idxmax()
top_county        = county.index[0]
top_age           = age["revenue"].idxmax()
peak_band         = tb["orders"].idxmax()
peak_day          = wd.idxmax()
champ_rev         = seg.loc["Champions", "revenue_share"]
champ_cust        = seg.loc["Champions", "customer_share"]
lapsed_n          = int((repeat_clv["status"] == "Lapsed").sum())
drifting_n        = int((repeat_clv["status"] == "Drifting").sum())
winback_rev       = repeat_clv.loc[repeat_clv["status"].isin(["Drifting", "Lapsed"]), "total_revenue"].sum()
at_risk_n         = int(seg.loc["At Risk", "customers"])
orders_chg        = ytd.loc["Change 2026 vs 2025", "Total orders"]
newcust_chg       = ytd.loc["Change 2026 vs 2025", "New customers"]
returning_chg     = ytd.loc["Change 2026 vs 2025", "Orders from returning customers"]

summary = [
 f"Scope: {n_customers:,} customers, {n_orders:,} orders, {first_dt:%b %Y} to {last_dt:%b %Y}; delivered revenue KES {delivered['net_revenue'].sum():,.0f}.",
 f"Retention: {repeat_rate:.0%} of customers have ordered more than once; the average customer places {customers['orders_placed'].mean():.1f} orders.",
 f"Value of loyalty: repeat customers are worth {ratio:.1f}× one-time customers and generate {compare.loc['Repeat','revenue_share']:.0%} of all revenue.",
 f"Concentration: Champions are {champ_cust:.0%} of scored customers but {champ_rev:.0%} of revenue; the top 20% of customers produce {top20_share:.0%}.",
 f"Channels: {best_loyalty_ch} acquires the most loyal customers (180-day repeat rate {scorecard.loc[best_loyalty_ch,'repeat_within_180d']:.0%}); "
 f"{best_aov_ch} has the highest order value (KES {scorecard.loc[best_aov_ch,'avg_order_value']:,.0f}); {worst_cancel_ch} has the highest cancellation rate ({scorecard.loc[worst_cancel_ch,'cancellation_rate']:.1%}).",
 f"Who/where: {top_county} leads revenue ({county.loc[top_county,'revenue_share']:.0%}); the {top_age} age group is the biggest revenue contributor ({age.loc[top_age,'revenue_share']:.0%}).",
 f"Growth engine: Jan–Oct 2026 orders are {orders_chg:+.0%} vs the same months of 2025, but new customers are {newcust_chg:+.0%} and orders from returning customers are {returning_chg:+.0%}, so existing customers are carrying the growth.",
 f"Timing: most orders arrive in the {peak_band} window and {peak_day} is the busiest weekday.",
 f"At Risk: {at_risk_n} customers scored 'At Risk' in RFM. Typical rhythm: the median gap between orders is {gaps['days_since_prev'].median():.0f} days; {drifting_n} repeat customers are drifting and {lapsed_n} have lapsed (>{churn_threshold:.0f} days quiet), together KES {winback_rev:,.0f} of past revenue.",
 f"Leakage: cancellations and returns removed {not_delivered.shape[0] / len(df):.1%} of orders and KES {not_delivered['order_total'].sum():,.0f} of order value.",
]
for i, line in enumerate(summary, 1):
    print(f"{i}. {line}\n")

**The story in one paragraph.** Immaculate Vibes runs on loyalty: six in ten customers come back, repeat customers are worth almost four times a one-time buyer, and a core group of Champions and Loyal customers (under half the customer base) produce about 70% of revenue. Orders are growing fast, but it's returning customers driving that growth while new-customer acquisition has slowed. The value of a repeat customer comes from them coming back more often, not from bigger baskets, and the hardest step is the second order, which typically takes five months. Meanwhile, about 140 repeat customers worth KES 1.27M of past revenue are drifting away.

### 9.2 Retention recommendations

1. **Shorten the path to the second order.** The typical wait is about 157 days and only 24% of customers reorder within 90. Set up a simple follow-up routine (for example on WhatsApp, at roughly day 30, 60 and 90) with a new-collection preview, a restock notice, or a small second-order perk. Start with the 62 *New/Promising* customers and compare their reorder rate with a group that gets no message.
2. **Look after the Champions and Loyal group.** About 44% of scored customers bring in 69% of revenue. Early access to new drops, a VIP tag, or a referral reward costs little compared with what they are worth, and it protects the most important income.
3. **Run a win-back campaign, worth-first.** Use `winback_list_repeat_customers.csv` (141 customers, sorted by lifetime value) together with the RFM *At Risk* segment (107 customers). Start with the *Drifting* group, because they are cheaper to win back than people gone for a year or more, and message them on WhatsApp since most repeat customers use several channels.
4. **Rebuild the new-customer pipeline, with a test.** New customers are down about 49% in 2026 versus 2025. TikTok remains the volume engine, but Instagram-acquired customers return the most and are worth the most. Try shifting a small slice of acquisition effort toward Instagram and compare cohorts before committing a large budget, since the sample sizes are small.
5. **Match marketing and operations to the demand pattern.** Concentrate posts and ads in the 17:00–21:00 window and on Saturdays; test a Sunday-evening promotion to wake up the quietest day; and prepare production capacity for a busy Nov–Dec. Separately, investigate why Palazzo pants have the highest cancellation rate (8.5%): fixing cancellations recovers up to 8% of order value.

### 9.3 Limitations: what to keep in mind

* **A recent customer hasn't had time to come back yet.** Simple repeat rates are understated for newer customers, so where it mattered we used "returned within 180 days" for customers old enough to measure. The repurchase-gap figures only include customers who *did* return.
* **The records cover 24 months of one business.** Patterns such as seasonality rest on two years of history, so conclusions should be rechecked as new orders arrive.
* **Some groups are small.** Walk-in (31 customers), Referral (75), and male customers (143 orders) are small enough that a gap of a few percentage points could be chance. We did not run significance tests.
* **Channel is credited to the first order only.** Over half of customers use more than one channel, so "acquisition channel" is a simplification of a more complex journey.
* **CLV here is historical, not predictive**, and based on revenue rather than profit. The data includes `cost_estimate` and `gross_profit_estimate`; repeating the CLV and channel analysis on profit, and adding acquisition cost per channel, would be the natural next step.
* **Segment rules are conventions.** The RFM cut-offs (and our choice to check *At Risk* before *Loyal*) and the 209/351-day drift and lapse thresholds are sensible, data-informed choices, but different cut-offs would shift the group sizes.
* **Cancelled orders show no money collected**, so we can't tell whether deposits were refunded or kept.

### 9.4 Files created by this notebook

| File | What it holds |
|---|---|
| `rfm_segments.csv` | Every customer with R/F/M scores, segment, and profile details |
| `winback_list_repeat_customers.csv` | Repeat customers who are drifting or lapsed, ranked by lifetime value |
| `figures/*.png` | Every chart in this notebook as a standalone image |

---
## 10. Deployment Phase – from analysis to a weekly Retention Action Engine

**Why a deployment phase?** Sections 1–9 answered *"what is happening?"* once. The strategy report's quick wins
(reorder nudges at days 30/60/90, a worth-first win-back list, measuring against a control group) only pay off if
somebody gets the right list **every week** without opening this notebook. So the analysis rules have been moved
into a small, tested Python package and two front ends:

| Notebook piece | Where it lives now | Who uses it |
|---|---|---|
| Load + quality checks (2.1–2.4) | `pipeline.load_orders`, `pipeline.quality_report` – a bad export is **stopped** before any list is produced | Automatic |
| Customer table (2.7) | `pipeline.build_customers` | Automatic |
| RFM + segment rules (4.1–4.3) | `pipeline.score_rfm`, `pipeline.assign_segment` (same rule order: *At Risk* before *Loyal*) | Sales champion |
| Repurchase gaps, drift/lapse (8.3–8.4) | `pipeline.thresholds`, `pipeline.lifecycle` (P75 / P90 recomputed on every run) | Sales champion |
| Recommendations 1 & 3 (9.2) | `pipeline.build_action_list` – nudge + win-back list with a 20% random **hold-out** group | Staff, weekly |
| Report Table 4 targets | `pipeline.compute_kpis` | Owner, weekly |
| Report §4.3 "does it pay?" | `pipeline.roi_scenario`, `pipeline.sensitivity_grid` | Owner |
| — | `run_pipeline.py` (batch job, scheduled weekly by GitHub Actions) and `app.py` (Streamlit app) | Everyone |

The design follows the report's choices: **batch, not real-time** (repurchase gaps are measured in months), **simple
explainable rules before models** (small data), **privacy by design** (pseudonymous IDs, consent flag, real data
never in git).

### 10.1 Parity check: does the production code give the same answers as this notebook?

Before anything is deployed, the package is run on the same file and its results are compared with the variables
computed above. If any `assert` fails, the deployment is not trustworthy and must not be used.

In [ ]:
from iv_retention import pipeline as ivp

prod = ivp.run(ivp.load_orders(DATA_PATH))

checks = {
    "Snapshot date":            (prod.snapshot == snapshot_date),
    "Customers scored (RFM)":   (len(prod.rfm) == len(rfm)),
    "Segment sizes":            prod.segments["customers"].astype(int).equals(seg["customers"].astype(int)),
    "Customer-level segment":   (prod.rfm.set_index("customer_id")["Segment"]
                                 .equals(rfm.set_index("customer_id")["Segment"])),
    "Drift threshold (days)":   np.isclose(prod.drift_days, drift_threshold),
    "Lapse threshold (days)":   np.isclose(prod.lapse_days, churn_threshold),
    "Win-back list members":    set(prod.lifecycle.query("status in ['Drifting','Lapsed']").index) == set(winback["customer_id"]),
    "Win-back past revenue":    np.isclose(prod.kpis["winback_pool_revenue"], winback_rev),
    "Repeat rate":              np.isclose(prod.kpis["repeat_rate"], repeat_rate),
    "2nd order within 90 days": np.isclose(prod.kpis["second_order_90d"], conv.loc[90, "placed_2nd_order"]),
}
parity = pd.Series(checks, name="matches notebook").to_frame()
display(parity)
assert parity["matches notebook"].all(), "Production code disagrees with the notebook – do not deploy"
print("✓ Production pipeline reproduces the notebook exactly.")

### 10.2 Validation: does the drift/lapse rule pick the right people? (back-test against a baseline)

The win-back list uses "days since last order" to decide who to contact. A list is only useful if it points
effort at customers who would **not come back on their own** – otherwise we pay for perks given to people who were
going to reorder anyway.

**How the test works (no information leakage):** freeze the data at a past date, compute the thresholds and statuses
using *only* orders before that date, then look 180 days ahead to see who actually came back, with no messaging in
between (none was being sent then). We repeat this at five dates and pool the results, and compare the deployed rule
against two baselines: *"message every repeat customer"* and the blueprint's naive *"silent for more than 14 days"*.

In [ ]:
AS_OF = ["2025-09-01", "2025-11-01", "2026-01-01", "2026-03-01", "2026-05-01"]
pooled, per_date = ivp.rolling_backtest(df, AS_OF, horizon_days=180)

display(pooled.rename(columns={"flagged": "customer-snapshots flagged",
                               "avg_share_flagged": "avg share of regulars flagged",
                               "returned_unaided_rate": "came back unaided within 180d"})
        .style.format({"avg share of regulars flagged": "{:.0%}", "came back unaided within 180d": "{:.0%}",
                       "ci_low": "{:.0%}", "ci_high": "{:.0%}", "returned": "{:,.0f}"}))

plot = pooled.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(plot["rule"], plot["returned_unaided_rate"], color=BRAND, height=0.55)
ax.errorbar(plot["returned_unaided_rate"], plot["rule"],
            xerr=[plot["returned_unaided_rate"] - plot["ci_low"], plot["ci_high"] - plot["returned_unaided_rate"]],
            fmt="none", ecolor="#333", capsize=3, lw=1)
ax.axvline(pooled.loc[0, "returned_unaided_rate"], ls="--", color="grey", lw=1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlim(0, 1)
ax.set_xlabel("Share who came back within 180 days with no message (95% interval)")
ax.set_title("About half of regulars return on their own, whatever rule flags them")
save_fig("10_backtest_rules")

> ### What the back-test tells us (and how it changed the deployment)
> * **"Days silent" does not predict who will stay away.** Across five back-test dates, roughly **55%** of repeat
>   customers came back within six months *without any message*, and the rate is similar whichever rule flagged
>   them. *Drifting* customers came back a little less often (≈47%) and *Lapsed* a little more (≈62%), but the
>   intervals overlap with the baseline, so we cannot claim the rule separates "leavers" from "stayers".
> * **The naive 14-day rule is useless as a list**: it flags ~86% of all regulars – it is just "everyone".
> * **What the deployed rule *does* do is focus effort.** Drifting + Lapsed is about a quarter of regulars, ordered
>   by past value, which is a list two people can actually work through in a week. That is a workload decision, not
>   a prediction.
> * **Consequence for deployment:** because about half would return anyway, the only honest way to know whether
>   messages work is to compare messaged customers with a **random hold-out group** that is not messaged. The
>   pipeline therefore holds back 20% of every list (deterministically, by a hash of the customer ID), and the app's
>   *"Did it work?"* tab measures uplift = messaged return rate − hold-out return rate.
> * **Why no churn model (yet):** a machine-learning model would need a signal that "days silent" doesn't carry, and
>   with ~330 regulars there is too little data to find one reliably. Per the report's Phase 3, revisit once the
>   hold-out results and more history exist. *Caveat:* the five snapshots overlap (the same customer can appear in
>   several), so the intervals are slightly narrower than they should be.

### 10.3 Is a win-back campaign worth it? Scenario analysis with sensitivity

The benefit depends on things we have not observed yet, so this is a **forecast range, not a result**. Two
assumptions matter most: the **uplift** (extra customers who return *because of* the message, above the ~55% who
return anyway) and the **gross margin**. Cost per contact (message + a free-delivery perk) is the third.

In [ ]:
n_contact = int(((prod.actions["action"].str.startswith("Win-back")) & (prod.actions["group"] == "Message")).sum())
aov_d = delivered["net_revenue"].mean()
gm = prod.kpis["gross_margin"]
base = ivp.roi_scenario(n_contact, response_rate=0.10, orders_per_win=2, aov=aov_d, margin=gm,
                        cost_per_contact=150, fixed_cost=5000)
print(f"Messaged: {n_contact} regulars | AOV KES {aov_d:,.0f} | gross margin {gm:.0%}")
print(f"Central case (10% uplift, 2 orders each): {base['customers_won']:.0f} customers won back, "
      f"KES {base['extra_revenue']:,.0f} extra revenue, KES {base['extra_gross_profit']:,.0f} gross profit, "
      f"net benefit KES {base['net_benefit']:,.0f}")
print(f"Break-even uplift: {base['break_even_response_rate']:.1%} "
      f"(≈{base['break_even_response_rate'] * n_contact:.1f} extra customers)")

grid = ivp.sensitivity_grid(n_contact, aov_d, 150, 5000, [0.02, 0.05, 0.10, 0.15, 0.20], [0.40, 0.45, 0.49, 0.55])
print("\nNet benefit (KES) – rows: gross margin, columns: uplift")
display(grid.style.format("{:,.0f}").format_index("{:.0%}", axis=0).format_index("{:.0%}", axis=1)
        .background_gradient(cmap="Purples"))

cost_sens = pd.DataFrame({c: ivp.roi_scenario(n_contact, 0.05, 2, aov_d, gm, c, 5000) for c in [50, 150, 300, 500]}).T
print("Cost per contact sensitivity at a cautious 5% uplift:")
display(cost_sens[["cost", "extra_gross_profit", "net_benefit", "break_even_response_rate"]]
        .style.format({"cost": "{:,.0f}", "extra_gross_profit": "{:,.0f}", "net_benefit": "{:,.0f}",
                       "break_even_response_rate": "{:.1%}"}))

> **Reading the scenarios.** At KES 150 per contact the campaign needs an uplift of about **6 percentage points** to
> break even (roughly 7 extra regulars out of ~110 messaged); the central 10% case gives a modest net gain of about
> KES 13,000 in gross profit. **The perk cost is the lever the owner controls:** at KES 50 per contact (a message plus
> a discount only paid when redeemed) break-even drops to about 3 points. So the advice is: start with cheap,
> redeem-only perks, and let the hold-out comparison show the real uplift within 2–3 months before spending more. These numbers are **gross profit, not cash**: deposits arrive with the order, but the perk cost is paid
> on redemption, and none of this covers the report's KES 350k 24-month analytics budget on its own.

### 10.4 This week's action list (the deployed output)

This is what staff receive every Monday – the same table the app shows and the batch job writes to
`outputs/action_list_<date>.csv`.

In [ ]:
acts = prod.actions
print(acts.groupby(["action", "group"]).size().unstack(fill_value=0))
print(f"\nTo message this week: {(acts['group'] == 'Message').sum()} customers "
      f"| hold-out (do not message): {(acts['group'] != 'Message').sum()}")
acts.to_csv(OUT_DIR / f"action_list_{prod.kpis['snapshot_date']}.csv", index=False)
acts.head(10)

### 10.5 Monitoring plan and decision gates

| KPI | Definition (computed by `compute_kpis`) | Baseline | Target / threshold | Review | Stop or revise if |
|---|---|---|---|---|---|
| **Primary: 2nd order within 90 days** | Customers whose 1st order was ≥90 days ago and who placed a 2nd within 90 days | 24% | 35% in 24 months; watch the *recent cohort* figure monthly | Monthly, owner | Recent cohort not above 24% after 6 months of nudges |
| Win-back uplift | Return rate of messaged − hold-out, per campaign | Unknown (≈55% return unaided) | > +5 points | Month 3 and 6 | Uplift ≤ 0 at month 6 → stop perks, keep plain messages |
| Win-back rate | Share of the 141 drifting/lapsed regulars who ordered again | 0 of 141 | 15% (≈21) | Monthly | — |
| Cancel + return rate | Non-delivered orders ÷ all orders, last 90 days | 8.7% | 6.5% | Monthly | Rises above 9% for 2 months |
| Opt-outs | STOP replies ÷ messages sent | — | < 5% | Weekly | > 5% → pause and rewrite messages |
| Model health | Drift/lapse thresholds vs last run; data-quality warnings | 209 / 351 days | Change < 20% | Each run | Big jump → check the export before using the list |

Every run appends to `outputs/kpi_history.csv`, so trends are visible without re-running this notebook. The
report's gates still apply: **month 6** – continue only if the list is used weekly and messages beat the hold-out;
**month 12** – continue only if the weekly job runs without manual fixes.

In [ ]:
# The same KPIs the app and batch job report, from the production code
pd.Series(prod.kpis).to_frame("value")

### 10.6 Responsible deployment

* **Data minimisation:** the engine needs only pseudonymous `customer_id`s and order fields. Phone numbers stay in
  WhatsApp Business; staff look a customer up there by ID. No names or numbers enter the app or the repo.
* **Consent (Data Protection Act 2019, s.37):** each action row carries `whatsapp_consent`. Until a consent register
  is uploaded it reads *"Unknown – confirm before messaging"*, and staff are instructed not to send marketing
  messages without consent. Every message includes an opt-out line; STOP replies go into the register the same day.
* **Access and storage:** the real export is git-ignored; the public repo holds only the de-identified
  `sample_data/`. The hosted app is password-protected (`APP_PASSWORD` secret) and keeps uploads only in memory for
  the session. Weekly outputs are kept 30 days as private GitHub Actions artifacts, then deleted.
* **Human in the loop:** the list *suggests*; a person decides whether and what to send. Nothing is sent
  automatically.
* **Known limits that could change the decision:** the rule does not predict churn (10.2); uplift is unmeasured until
  the hold-out results arrive; segment thresholds are conventions; CLV is historical revenue, not profit after
  acquisition cost.

### 10.7 How to run it

```bash
pip install -r requirements.txt
python run_pipeline.py --data data/Immaculate_Vibes_Sales.csv --snapshot today   # weekly batch job
streamlit run app.py                                                           # owner/staff app
pytest -q                                                                      # tests incl. parity
```
See `README.md` for hosting on Streamlit Community Cloud and the GitHub Actions schedule.